# 04 — Running a diagnostic locally

The earlier notebooks read *pre-computed* results from the public API. This notebook does
the opposite: it runs a REF diagnostic **locally**, on your own machine, against a small
sample dataset — so you can see the evaluation pipeline end to end.

**Prerequisites:** [01 — REF concepts](01-ref-concepts.ipynb).

**What you need:** the `climate-ref` packages (installed by `uv sync`) and an internet
connection the first time, to download the sample data. The sample data is intentionally
tiny — this runs comfortably on Binder.

## 1. Fetch the sample data

The REF ships a *decimated* CMIP6 sample dataset for testing and demonstration. The
`ref_tutorials` helper wraps the REF CLI's fetch so we get it with one call.

In [1]:
from ref_tutorials import fetch_sample_data

sample_data_dir = fetch_sample_data()
sample_data_dir

/Users/jared/code/climate-ref/climate-ref-tutorials/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/Users/jared/code/climate-ref/climate-ref-tutorials/.venv/lib/python3.14/site-packages/rich/live.py:260: 
UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

2026-05-22 16:11:01.953 | INFO     | climate_ref_core.dataset_registry:fetch_all_files:171 - File /Users/jared/code/tests/test-data/sample-data/CMIP6/C4MIP/CSIRO/ACCESS-ESM1-5/esm-1pct-brch-1000PgC/r1i1p1f1/Amon/tas/gn/v20191206/tas_Amon_ACCESS-ESM1-5_esm-1pct-brch-1000PgC_r1i1p1f1_gn_016801-026812.nc already exists. Skipping.
2026-05-22 16:11:01.957 | INFO     | climate_ref_core.dataset_registry:fetch_all_files:171 - File /Users/jared/code/tests/test-data/sample-data/CMIP6/C4MIP/CSIRO/ACCESS-ESM1-5/esm-1pct-brch-1000PgC/r1i1p1f1/fx/areacella/gn/v20191206/areacella_fx_ACCESS-ESM1-5_esm-1pct-brch-1000PgC_r1i1p1f1_gn.nc already exists. Skipping.
2026-05-22 16:11:01.959 | INFO     | climate_ref_core.dataset_registry:fetch_all_files:171 - File /Users/jared/code/tests/test-data/sample-data/CMIP6/C4MIP/MPI-M/MPI-ESM1-2-LR/esm-1pctCO2/r1i1p1f1/Amon/fco2antt/gn/v20190815/fco2antt_Amon_MPI-ESM1-2-LR_esm-1pctCO2_r1i1p1f1_gn_185001-186912.nc already exists. Skipping.
2026-05-22 16:11:01.961 | INF

PosixPath('/Users/jared/code/tests/test-data/sample-data')

## 2. A custom diagnostic provider

Providers can be used directly, without the REF database or web infrastructure. Instead
of a built-in provider, the tutorials define their own minimal one in
`src/ref_tutorials/provider.py` — a real, self-contained custom provider you can read
and adapt. Its single diagnostic, `annual-mean-global-mean-tas`, computes the
annual-mean, area-weighted global-mean near-surface air temperature (`tas`) of a CMIP6
dataset.

Open `src/ref_tutorials/provider.py` alongside this notebook: a diagnostic is a class
with `data_requirements` (what data it needs), `execute` (the calculation), and
`build_execution_result` (packaging the output).

In [2]:
from climate_ref.config import Config

from ref_tutorials.provider import provider

config = Config.default()
provider.configure(config)

diagnostic = provider.get("annual-mean-global-mean-tas")
diagnostic.slug

2026-05-22 16:11:03.090 | DEBUG    | climate_ref.config:default:554 - Loading default configuration from /Users/jared/Library/Application Support/climate_ref/ref.toml
2026-05-22 16:11:03.097 | DEBUG    | climate_ref_core.providers:configure:82 - Configuring provider tutorial using ignore_datasets_file /Users/jared/Library/Caches/climate_ref/default_ignore_datasets.yaml


'annual-mean-global-mean-tas'

## 3. Build a data catalog from disk

A **data catalog** is a table of available datasets and their metadata. We can build one
directly from the sample-data directory — no database required — using the CMIP6
dataset adapter.

In [3]:
from climate_ref.datasets import get_dataset_adapter

adapter = get_dataset_adapter("cmip6")
data_catalog = adapter.find_local_datasets(sample_data_dir / "CMIP6")

data_catalog[["source_id", "variant_label", "variable_id", "experiment_id"]].drop_duplicates()

2026-05-22 16:11:03.125 | DEBUG    | climate_ref.config:default:554 - Loading default configuration from /Users/jared/Library/Application Support/climate_ref/ref.toml
2026-05-22 16:11:03.129 | INFO     | climate_ref.datasets.cmip6:get_parsing_function:178 - Using complete CMIP6 parser
2026-05-22 16:11:03.143 | INFO     | climate_ref.datasets.catalog_builder:build_catalog:154 - Discovered 102 files matching ['*.nc'] in ['/Users/jared/code/tests/test-data/sample-data/CMIP6']
Parsing files: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 102/102 [00:00<00:00, 703.00file/s]
2026-05-22 16:11:03.298 | INFO     | climate_ref.datasets.catalog_builder:build_catalog:160 - Built catalog with 102 valid entries (0 invalid)


,source_id,variant_label,variable_id,experiment_id
0,ACCESS-ESM1-5,r1i1p1f1,tas,esm-1pct-brch-1000PgC
1,ACCESS-ESM1-5,r1i1p1f1,areacella,esm-1pct-brch-1000PgC
2,MPI-ESM1-2-LR,r1i1p1f1,fco2antt,esm-1pctCO2
6,MPI-ESM1-2-LR,r1i1p1f1,tas,esm-1pctCO2
10,CanESM5,r1i1p1f1,tas,historical
...,...,...,...,...
97,ACCESS-ESM1-5,r1i1p1f1,rsdt,ssp126
98,ACCESS-ESM1-5,r1i1p1f1,rsut,ssp126
99,ACCESS-ESM1-5,r1i1p1f1,tas,ssp126
100,ACCESS-ESM1-5,r1i1p1f1,tos,ssp126


## 4. Solve for executions

Given the diagnostic's data requirements and the available datasets, the **solver**
works out which *executions* are possible. Each execution pairs the diagnostic with one
group of datasets.

In [4]:
from climate_ref.solver import solve_executions
from climate_ref_core.datasets import SourceDatasetType

executions = list(
    solve_executions(
        data_catalog={SourceDatasetType.CMIP6: data_catalog},
        diagnostic=diagnostic,
        provider=provider,
    )
)
print(f"{len(executions)} executions proposed")

10 executions proposed


## 5. Run one execution

An execution needs an `ExecutionDefinition` — it says which datasets to use and where to
write output. We build one for the first execution and run the diagnostic directly.

In [5]:
from pathlib import Path

output_directory = Path("output") / "local-run"
definition = executions[0].build_execution_definition(output_directory)
definition.output_directory.mkdir(parents=True, exist_ok=True)

result = diagnostic.run(definition=definition)
assert result.successful, "diagnostic run failed"

/Users/jared/code/climate-ref/climate-ref-tutorials/src/ref_tutorials/provider.py:67: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  ds = xr.open_mfdataset(
/Users/jared/code/climate-ref/climate-ref-tutorials/src/ref_tutorials/provider.py:67: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  ds = xr.open_mfdataset(


## 6. Inspect the output

The run wrote an output bundle into the output directory. Let's read it back.

In [6]:
import json

output_file = definition.to_output_path("output.json")
print(json.dumps(json.loads(output_file.read_text()), indent=2)[:2000])

{
  "index": "index.html",
  "provenance": {
    "environment": {},
    "modeldata": [],
    "obsdata": {},
    "log": "cmec_output.log"
  },
  "data": {},
  "plots": {},
  "html": {},
  "metrics": null,
  "diagnostics": {}
}


## Recap

- A **custom provider** — defined here in `src/ref_tutorials/provider.py` — runs a
  diagnostic locally with no database or web infrastructure.
- `find_local_datasets` builds a data catalog straight from a directory.
- `solve_executions` turns *(diagnostic + catalog)* into concrete executions.
- `build_execution_definition` + `diagnostic.run` executes one of them.

This is the same machinery the full REF uses — just driven by hand. Writing your own
provider is how you add a new evaluation to the REF.

You have completed the training set. To go deeper, browse the REF documentation at
<https://climate-ref.org> and the API docs at <https://api.climate-ref.org/docs>.